# Analyse des élus municipaux 2026


## Initialisation

In [4]:
# Importation des modules et fichiers
import pandas as pd
import numpy as np
elus_epci = pd.read_csv("elus-conseillers-communautaires-epci.csv", sep=";", encoding="UTF-8",dtype=str)
elus_2026 = pd.read_csv("elus-conseillers-municipaux-cm.csv", sep=";", encoding="UTF-8",dtype=str)
elus_sortants = pd.read_csv("mun2026-cm-sortants-20260227.csv", sep=";", encoding="UTF-8",dtype=str)

# elus_epci.head()
# elus_epci.info()
# print(elus_epci.columns.tolist())
# print(elus_2026.columns.tolist())
# print(elus_sortants.columns.tolist())

# Création d'une colonne d'identifiant

elus_epci["id_unique"] = (
    elus_epci["Prénom de l'élu"].str.strip().str.upper() + "_" +
    elus_epci["Nom de l'élu"].str.strip().str.upper() + "_" +
    elus_epci["Libellé de la commune de rattachement"].str.strip().str.upper()
)
elus_2026["id_unique"] = (
    elus_2026["Prénom de l'élu"].str.strip().str.upper() + "_" +
    elus_2026["Nom de l'élu"].str.strip().str.upper() + "_" +
    elus_2026["Libellé de la commune"].str.strip().str.upper()
)
elus_sortants["id_unique"] = (
    elus_sortants["Prénom de l'élu"].str.strip().str.upper() + "_" +
    elus_sortants["Nom de l'élu"].str.strip().str.upper() + "_" +
    elus_sortants["Libellé de la commune"].str.strip().str.upper()
)

In [2]:
# Base : les élus municipaux actuels
elus_municipaux = elus_2026[["Prénom de l'élu", "Nom de l'élu", "Libellé de la commune", "Code de la commune", "Libellé de la fonction", "id_unique", "Code du département"]].copy()

# Était-il élu au dernier mandat ?
elus_municipaux["Elu au mandat précédent"] = elus_municipaux["id_unique"].isin(elus_sortants["id_unique"])

# Est-il élu à la communauté de communes ?
elus_municipaux["Elu à la communauté de communes"] = elus_municipaux["id_unique"].isin(elus_epci["id_unique"])

# Nom de la communauté de communes (récupéré via une jointure)
epci_info = elus_epci[["id_unique", "Libellé de l'EPCI"]].drop_duplicates(subset="id_unique")

elus_municipaux = elus_municipaux.merge(epci_info, on="id_unique", how="left")


## Séléction de la communauté de commune

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Extraction du département à partir du code commune
elus_municipaux["Code departement"] = elus_municipaux["Code de la commune"].str[:2]

# Liste des départements disponibles
liste_dept = sorted(elus_municipaux["Code departement"].dropna().unique())

# Widgets
dropdown_dept = widgets.Dropdown(
    options=liste_dept,
    value="44",  # ← département par défaut : Loire Atlantique
    description="Département :",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)

dropdown_epci = widgets.Dropdown(
    description="EPCI :",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

case_epci = widgets.Checkbox(
    value=False,
    description="Uniquement les élus à l'EPCI"
)

output = widgets.Output()
output_synthese = widgets.Output()

# Met à jour la liste des EPCI en fonction du département choisi
def maj_epci(change):
    epci_filtres = sorted(
        elus_municipaux[elus_municipaux["Code departement"] == dropdown_dept.value]["Libellé de l'EPCI"].dropna().unique()
    )
    dropdown_epci.options = epci_filtres

# Calcule la synthèse par commune, limitée à l'EPCI sélectionné
def afficher_synthese(selection):
    with output_synthese:
        clear_output()
        communes_epci = elus_epci[elus_epci["Libellé de l'EPCI"] == selection]["Code de la commune de rattachement"].unique()
        data_epci = elus_municipaux[elus_municipaux["Code de la commune"].isin(communes_epci)]

        synthese_commune = data_epci.groupby(["Libellé de la commune", "Code de la commune"]).agg(
            nb_elus=("id_unique", "count"),
            nb_nouveaux=("Elu au mandat précédent", lambda x: (~x).sum()),
            nb_elus_epci=("Elu à la communauté de communes", "sum"),
        ).reset_index()

        nouveaux_epci = data_epci[data_epci["Elu à la communauté de communes"] == True].groupby(
            ["Libellé de la commune", "Code de la commune"]
        ).agg(
            nb_nouveaux_epci=("Elu au mandat précédent", lambda x: (~x).sum())
        ).reset_index()

        synthese_commune = synthese_commune.merge(nouveaux_epci, on=["Libellé de la commune", "Code de la commune"], how="left")
        
        synthese_commune["nb_nouveaux_epci"] = synthese_commune["nb_nouveaux_epci"].fillna(0).astype(int)
        synthese_commune["pct_nouveaux"] = (synthese_commune["nb_nouveaux"] / synthese_commune["nb_elus"] * 100).round(0).astype(int)
       
        synthese_commune["pct_nouveaux_epci"] = (
            synthese_commune["nb_nouveaux_epci"] / synthese_commune["nb_elus_epci"] * 100
        ).round(0).fillna(0).astype(int)

        # --- Ligne de total ---
        total = {
            "Libellé de la commune": "TOTAL",
            "Code de la commune": "",
            "nb_elus": synthese_commune["nb_elus"].sum(),
            "nb_nouveaux": synthese_commune["nb_nouveaux"].sum(),
            "nb_elus_epci": synthese_commune["nb_elus_epci"].sum(),
            "nb_nouveaux_epci": synthese_commune["nb_nouveaux_epci"].sum(),
        }

        total["pct_nouveaux"] = round(total["nb_nouveaux"] / total["nb_elus"] * 100, 0) if total["nb_elus"] > 0 else 0
        total["pct_nouveaux_epci"] = round(total["nb_nouveaux_epci"] / total["nb_elus_epci"] * 100, 0) if total["nb_elus_epci"] > 0 else 0

        
        synthese_commune = pd.concat([synthese_commune, pd.DataFrame([total])], ignore_index=True)

        # --- Intervertie les colonnes pct

        # --- Catégories au-dessus des colonnes ---
        synthese_commune.columns = pd.MultiIndex.from_tuples([
            ("", "Libellé de la commune"),
            ("", "Code de la commune"),
            ("Communes", "nb_elus"),
            ("Communes", "nb_nouveaux"),
            ("EPCI", "nb_elus_epci"),
            ("EPCI", "nb_nouveaux_epci"),
            ("Pourcentage nouveaux","Communes"),
            ("Pourcentage nouveaux","EPCI"),
        ])

        display(synthese_commune)
        
# Affiche les élus en fonction de l'EPCI choisi (et de la case à cocher)
def afficher_elus(change):
    selection = dropdown_epci.value
    
    with output:
        clear_output()
        resultat = elus_municipaux[elus_municipaux["Libellé de l'EPCI"] == selection]
        
        if case_epci.value:
            resultat = resultat[resultat["Elu à la communauté de communes"] == True]
        
        display(resultat)
    
    afficher_synthese(selection)

# Connexions des widgets
dropdown_dept.observe(maj_epci, names="value")
dropdown_epci.observe(afficher_elus, names="value")
case_epci.observe(afficher_elus, names="value")

# Affichage
display(dropdown_dept, dropdown_epci, case_epci, output_synthese, output)

# Initialisation
maj_epci(None)
afficher_elus(None)

Dropdown(description='Département :', layout=Layout(width='300px'), options=('10', '11', '12', '13', '14', '15…

Dropdown(description='EPCI :', layout=Layout(width='500px'), options=(), style=DescriptionStyle(description_wi…

Checkbox(value=False, description="Uniquement les élus à l'EPCI")

Output()

Output()

In [10]:
elus_2026.head()

,Code du département,Libellé du département,Code de la collectivité à statut particulier,Libellé de la collectivité à statut particulier,Code de la commune,Libellé de la commune,Nom de l'élu,Prénom de l'élu,Code sexe,Date de naissance,Code de la catégorie socio-professionnelle,Libellé de la catégorie socio-professionnelle,Date de début du mandat,Libellé de la fonction,Date de début de la fonction,Code nationalité,id_unique
0,1,Ain,NaN,NaN,1001,L'Abergement-Clémenciat,ARMANDO,Chantal,F,1972-06-02,37,Cadre administratif et commercial d'entreprise,2026-03-15,NaN,NaN,FR,CHANTAL_ARMANDO_L'ABERGEMENT-CLÉMENCIAT
1,1,Ain,NaN,NaN,1001,L'Abergement-Clémenciat,BEAUDET,Sylvie,F,1967-03-25,12,Agriculteur sur moyenne exploitation,2026-03-15,NaN,NaN,FR,SYLVIE_BEAUDET_L'ABERGEMENT-CLÉMENCIAT
2,1,Ain,NaN,NaN,1001,L'Abergement-Clémenciat,BECHARD,Marie-Chantal,F,1962-10-04,74,Ancien cadre,2026-03-15,NaN,NaN,FR,MARIE-CHANTAL_BECHARD_L'ABERGEMENT-CLÉMENCIAT
3,1,Ain,NaN,NaN,1001,L'Abergement-Clémenciat,BOUILLOUX,Delphine,F,1977-08-02,47,Technicien,2026-03-15,2ème adjoint au Maire,2026-03-20,FR,DELPHINE_BOUILLOUX_L'ABERGEMENT-CLÉMENCIAT
4,1,Ain,NaN,NaN,1001,L'Abergement-Clémenciat,CASSAN,Pierre,M,1976-04-08,34,"Professeur, profession scientifique",2026-03-15,NaN,NaN,FR,PIERRE_CASSAN_L'ABERGEMENT-CLÉMENCIAT
